In [1]:
from gensim import corpora
from gensim import models
from gensim import similarities
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import snowball
import re

def my_tokenizer(text):
    """tokenization function"""
    sw=stopwords.words('english')
    stemmer=snowball.SnowballStemmer(language="english")
    tokens=word_tokenize(text)
    pruned=[stemmer.stem(t.lower()) for t in tokens \
            if re.search(r"^[a-zA-Z]",t) and not t.lower() in sw]
    return pruned

documents=["Indian government goes for open source software",
"Debian 3.0 Woody released",
"Wine 2.0 released with fixes for Gentoo 1.4 and Debian 3.0",
"gnuPOD released: iPOD on Linux… with GPLed software",
"Gentoo servers running at open source mySQL database",
"Dolly the sheep not totally identical clone",
"DNA news: introduced low-cost human genome DNA chip",
"Malaria-parasite genome database on the Web",
"UK sets up genome bank to protect rare sheep breeds",
"Dolly's DNA damaged"]

In [2]:
texts=[]
for d in documents:
    # creates an array of tokenized documents
    texts.append(my_tokenizer(d))

#creates the dictionary for the document corpus
dictionary = corpora.Dictionary(texts)
#creates a bag of word corpus
bow_corpus=[dictionary.doc2bow(text) for text in texts]

#creates a tf-idf model from the bag of word corpus
tfidf = models.TfidfModel(bow_corpus)

#extracts the tf-idf corpus
corpus_tfidf = tfidf[bow_corpus]

In [3]:
#creates the LSI model, with 2 topics
lsi_model=models.LsiModel(corpus_tfidf,num_topics=2,id2word=dictionary)

In [4]:
#creates an index that facilitates the computation of similarities
index = similarities.MatrixSimilarity(lsi_model[corpus_tfidf])

In [5]:
print(list(index)[0])

[1.0000001  0.97902197 0.9843578  0.9853968  0.99798703 0.10885489
 0.13524207 0.68249047 0.19551672 0.11114562]


In [6]:
print(list(index)[5])

[ 0.10885489 -0.09597263 -0.06798209 -0.06199703  0.17167735  1.
  0.9996466   0.8008437   0.9961556   0.9999973 ]


In [7]:
#tokenizes the query
query_document = my_tokenizer("DNA")

#indexes the query using the documents' dictionary
query_bow = dictionary.doc2bow(query_document)
query_lsi = lsi_model[query_bow]  # convert the query to LSI space

In [8]:
sims = index[query_lsi]
print(list(enumerate(sims)))

[(0, 0.118368536), (1, -0.08643689), (2, -0.05842559), (3, -0.05243706), (4, 0.18110284), (5, 0.9999541), (6, 0.99985534), (7, 0.8065415), (8, 0.9969488), (9, 0.99997354)]
